In [5]:
# Carregamento dos dados

import pandas as pd
import duckdb

arquivo = "yellow_tripdata_2026-01.parquet"
df = pd.read_parquet(arquivo)

con = duckdb.connect()
con.execute(f"""
   CREATE TABLE corridas AS
   SELECT * FROM read_parquet('{arquivo}')
""")

# Inspecionar o número de linhas, as colunas e os tipos de dados
con.table("corridas").show()
df = con.table("corridas").df()
df.info()

┌──────────┬──────────────────────┬───────────────────────┬─────────────────┬───────────────┬────────────┬────────────────────┬──────────────┬──────────────┬──────────────┬─────────────┬────────┬─────────┬────────────┬──────────────┬───────────────────────┬──────────────┬──────────────────────┬─────────────┬────────────────────┐
│ VendorID │ tpep_pickup_datetime │ tpep_dropoff_datetime │ passenger_count │ trip_distance │ RatecodeID │ store_and_fwd_flag │ PULocationID │ DOLocationID │ payment_type │ fare_amount │ extra  │ mta_tax │ tip_amount │ tolls_amount │ improvement_surcharge │ total_amount │ congestion_surcharge │ Airport_fee │ cbd_congestion_fee │
│  int32   │      timestamp       │       timestamp       │      int64      │    double     │   int64    │      varchar       │    int32     │    int32     │    int64     │   double    │ double │ double  │   double   │    double    │        double         │    double    │        double        │   double    │       double       │
├──────

### Questão 1

**Corridas por horário (sem IA):** Para cada hora de início da corrida (0 a 23, a partir de
tpep_pickup_datetime), calcule o número de corridas, a distância média (trip_distance)
e o valor médio pago (total_amount). Ordene pelo número de corridas, do maior para o
menor. Utilize os registros de janeiro de 2026.

In [6]:
# DuckDB
df_duckdb_hora_inicio = con.execute(f"""
    SELECT 
        HOUR(tpep_pickup_datetime) AS hora_inicio_corrida,
        COUNT(*) AS num_corridas,
        AVG(trip_distance) AS distancia_media,
        AVG(total_amount) AS valor_medio_pago
    FROM read_parquet('{arquivo}')
    GROUP BY hora_inicio_corrida
    ORDER BY num_corridas DESC
""").df()

print(df_duckdb_hora_inicio)

    hora_inicio_corrida  num_corridas  distancia_media  valor_medio_pago
0                    18        265574         3.191099         29.317995
1                    17        252173         6.149070         30.089539
2                    19        234542         4.336286         29.063146
3                    16        227893         4.936715         30.851314
4                    15        221677         5.255241         29.340657
5                    20        212883         5.726825         29.116929
6                    21        212544         7.311580         29.230308
7                    14        207507         6.578414         29.071705
8                    13        195708         5.526828         27.888121
9                    22        194801         5.441168         29.510664
10                   12        187435         5.295283         27.497725
11                   11        171729         7.003858         27.032998
12                   10        161419         9.127

In [7]:
# Pandas

df['tpep_pickup_datetime'] = pd.to_datetime(df['tpep_pickup_datetime'])

df_pandas_hora_inicio = (
    df.groupby(df['tpep_pickup_datetime'].dt.hour).agg(
                                                        num_corridas=('tpep_pickup_datetime', 'count'),
                                                        distancia_media=('trip_distance', 'mean'),
                                                        valor_medio_pago=('total_amount', 'mean'),
                                                    ).reset_index(names='hora_inicio_corrida').sort_values('num_corridas', ascending=False)
)

print(df_pandas_hora_inicio)

    hora_inicio_corrida  num_corridas  distancia_media  valor_medio_pago
18                   18        265574         3.191099         29.317995
17                   17        252173         6.149070         30.089539
19                   19        234542         4.336286         29.063146
16                   16        227893         4.936715         30.851314
15                   15        221677         5.255241         29.340657
20                   20        212883         5.726825         29.116929
21                   21        212544         7.311580         29.230308
14                   14        207507         6.578414         29.071705
13                   13        195708         5.526828         27.888121
22                   22        194801         5.441168         29.510664
12                   12        187435         5.295283         27.497725
11                   11        171729         7.003858         27.032998
10                   10        161419         9.127

# Questão 2

**Corridas semelhantes (com IA)**: Encontre pares de corridas distintas que tenham
a mesma origem (PULocationID), o mesmo destino (DOLocationID) e horários de início
(tpep_pickup_datetime) com diferença de, no máximo, cinco minutos. Conte cada par
apenas uma vez e apresente somente a quantidade total de pares. Para limitar o
volume, considere as corridas iniciadas em 15/01/2026, das 8h às 9h (sem incluir 9h).

In [8]:
import pandas as pd
import duckdb

# Janela de tempo (início inclusive, fim exclusivo)
inicio = "2026-01-15 08:00:00"
fim = "2026-01-15 09:00:00"


# ---------------------------------------------------------
# Implementação com Pandas
# ---------------------------------------------------------
def pares_pandas(df):
    # 1) Filtra as corridas da janela de tempo
    mascara = (df["tpep_pickup_datetime"] >= inicio) & (df["tpep_pickup_datetime"] < fim)
    janela = df.loc[mascara, ["PULocationID", "DOLocationID", "tpep_pickup_datetime"]]

    # 2) Guarda o índice original para identificar cada corrida
    janela = janela.reset_index().rename(columns={"index": "id"})

    # 3) Auto-junção: mesma origem e mesmo destino
    juntas = janela.merge(
        janela,
        on=["PULocationID", "DOLocationID"],
        suffixes=("_a", "_b"),
    )

    # 4) id_a < id_b: corridas distintas e cada par contado uma única vez
    juntas = juntas[juntas["id_a"] < juntas["id_b"]]

    # 5) Diferença de horário de no máximo 5 minutos
    diferenca = (juntas["tpep_pickup_datetime_a"] - juntas["tpep_pickup_datetime_b"]).abs()
    juntas = juntas[diferenca <= pd.Timedelta(minutes=5)]

    return len(juntas)


# ---------------------------------------------------------
# Implementação com DuckDB
# ---------------------------------------------------------
def pares_duckdb(con):
    consulta = f"""
        WITH janela AS (
            SELECT
                rowid AS id,
                PULocationID,
                DOLocationID,
                tpep_pickup_datetime
            FROM corridas
            WHERE tpep_pickup_datetime >= TIMESTAMP '{inicio}'
              AND tpep_pickup_datetime <  TIMESTAMP '{fim}'
        )
        SELECT COUNT(*) AS total_pares
        FROM janela a
        JOIN janela b
          ON  a.PULocationID = b.PULocationID
          AND a.DOLocationID = b.DOLocationID
          AND a.id < b.id
          AND b.tpep_pickup_datetime BETWEEN a.tpep_pickup_datetime - INTERVAL 5 MINUTE
                                         AND a.tpep_pickup_datetime + INTERVAL 5 MINUTE
    """
    return con.execute(consulta).fetchone()[0]


# ---------------------------------------------------------
# Execução e comparação
# ---------------------------------------------------------
total_pandas = pares_pandas(df)
total_duckdb = pares_duckdb(con)

print("Pandas:", total_pandas)
print("DuckDB:", total_duckdb)

assert total_pandas == total_duckdb, "Os resultados são diferentes!"

Pandas: 3721
DuckDB: 3721


Sugestões para as perguntas 3 e 4:

* Quais são os X lugares de embarque com maior número de corridas? (usar coluna PULocationID)
* Quais são os X destinos com maior valor médio por corrida? ()
* Como a duração média das corridas varia ao longo do dia? ps: IA
* Quais locais de origem E destino são mais frequentes em corridas? ps: IA de novo